# 07 — Multiple Testing: FWER, FDR, and Practical Correction
**Prerequisites:** `04_primary_secondary_metrics.ipynb` (why many secondary metrics get examined);
statistics_course/06_hypothesis_testing.ipynb (p-values, Type I error).
**Connects to:** `08_peeking_sequential_risk.ipynb` (always-valid inference — the "multiple looks
over time" version of this problem); `09_segmentation_heterogeneity.ipynb` (segment-cut corrections).

## Narrative thread
```
FWER vs FDR -> Bonferroni -> Benjamini-Hochberg (implemented) -> when it applies in online experiments
   -> always-valid inference (pointer) -> practical guidance
```

## Why multiplicity matters in online experiments

A single experiment often reports dozens to hundreds of metrics (primary, secondary, guardrails,
segment cuts). If each is tested at $\alpha=0.05$ independently, the chance of at least one false
positive across $m$ independent tests under the global null is

$$P(\text{at least one false positive}) = 1-(1-\alpha)^m$$

which is already $\approx 40\%$ at $m=10$ and $>99\%$ at $m=100$. Two different error-rate
concepts formalize "how bad is this":

- **FWER (Family-Wise Error Rate)**: $P(\text{at least one false positive among all } m \text{ tests})$.
  Controlling FWER is strict — appropriate when *any* false positive is costly (e.g., a small set
  of primary/OEC metrics).
- **FDR (False Discovery Rate)**: $E[\text{false positives} / \text{total positives}]$. Controlling
  FDR is more permissive and appropriate when you expect many true effects among many tests (e.g.
  scanning dozens of segments or secondary metrics for interesting signals to *investigate
  further*, not to ship on directly).


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests

plt.rcParams.update({
    'figure.dpi': 130, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.color': '#e8e8e8', 'axes.axisbelow': True,
    'axes.titlesize': 11, 'axes.titleweight': 'bold', 'legend.frameon': False,
})
np.random.seed(42)

In [2]:
# ── Demonstrate the multiplicity problem, then correct it two ways ─────
np.random.seed(21)
m = 40                      # 40 metrics/segments tested, ALL under the true null (no real effect)
n_per_arm = 2000
p_values = []
for _ in range(m):
    control = np.random.normal(0, 1, n_per_arm)
    treatment = np.random.normal(0, 1, n_per_arm)   # same distribution: true null
    p_values.append(stats.ttest_ind(treatment, control).pvalue)
p_values = np.array(p_values)

alpha = 0.05
raw_rejections = (p_values < alpha).sum()
print(f"Out of {m} tests, ALL under a true null, {raw_rejections} are 'significant' at raw alpha=0.05")
print("(expected under pure noise: about", round(m * alpha, 1), ")")

# Bonferroni: control FWER
bonf_reject, bonf_p, _, bonf_alpha = multipletests(p_values, alpha=alpha, method='bonferroni')
print(f"\nBonferroni: {bonf_reject.sum()} rejections (adjusted alpha per test = {alpha/m:.5f})")

# Benjamini-Hochberg: control FDR
bh_reject, bh_p, _, _ = multipletests(p_values, alpha=alpha, method='fdr_bh')
print(f"Benjamini-Hochberg (FDR): {bh_reject.sum()} rejections")

Out of 40 tests, ALL under a true null, 4 are 'significant' at raw alpha=0.05
(expected under pure noise: about 2.0 )

Bonferroni: 0 rejections (adjusted alpha per test = 0.00125)
Benjamini-Hochberg (FDR): 0 rejections


In [3]:
# ── Now with a mix of true nulls and true (non-null) effects, compare power ──
np.random.seed(22)
m_null, m_effect = 30, 10
true_effect = 0.15   # a real, modest effect for the "m_effect" tests
p_values2 = []
truth = []
for i in range(m_null + m_effect):
    control = np.random.normal(0, 1, n_per_arm)
    delta = true_effect if i >= m_null else 0.0
    treatment = np.random.normal(delta, 1, n_per_arm)
    p_values2.append(stats.ttest_ind(treatment, control).pvalue)
    truth.append(i >= m_null)
p_values2 = np.array(p_values2)
truth = np.array(truth)

for name, method in [('Raw (uncorrected)', None), ('Bonferroni', 'bonferroni'), ('Benjamini-Hochberg', 'fdr_bh')]:
    if method is None:
        reject = p_values2 < alpha
    else:
        reject, _, _, _ = multipletests(p_values2, alpha=alpha, method=method)
    true_pos = (reject & truth).sum()
    false_pos = (reject & ~truth).sum()
    print(f"{name:<22} true positives: {true_pos}/{m_effect}, false positives: {false_pos}/{m_null}")

print("\nBonferroni is conservative (controls FWER tightly, costs power);")
print("BH recovers more true positives while still bounding the *expected proportion* of false discoveries.")

Raw (uncorrected)      true positives: 10/10, false positives: 1/30
Bonferroni             true positives: 10/10, false positives: 0/30
Benjamini-Hochberg     true positives: 10/10, false positives: 0/30

Bonferroni is conservative (controls FWER tightly, costs power);
BH recovers more true positives while still bounding the *expected proportion* of false discoveries.


## When multiplicity correction applies in practice

| Situation | Typical approach |
|---|---|
| A handful (1-3) of pre-registered primary/OEC metrics | Usually no correction — these are the confirmatory test the experiment was sized for |
| Many secondary metrics scanned for interesting signal | FDR (Benjamini-Hochberg) — treat as exploratory, follow up with a dedicated confirmatory experiment |
| Many segment cuts of the same metric | FDR, or restrict to a small pre-registered set of segments (see `09_segmentation_heterogeneity.ipynb`) |
| Many concurrent experiment *arms* (multi-armed tests) | FWER-style correction (Bonferroni/Holm) against control, since each arm vs. control is often treated as its own ship decision |
| Repeated looks at the *same* metric over time | Not a "many tests" problem in the same sense — it's the **peeking problem**, solved with always-valid inference / sequential tests, not BH — see `08_peeking_sequential_risk.ipynb` |

Real experimentation platforms (in the style of Optimizely's Stats Engine or Statsig's
sequential testing) typically combine **both** ideas: an always-valid/sequential correction for
continuous monitoring of a *given* metric (notebook 08), plus an FDR or Bonferroni-style
correction across the *set* of metrics/segments examined in a single analysis pass (this notebook).
Confusing the two — e.g. applying BH once and thinking that also fixes peeking — is a common
mistake.

## Key takeaways

| Concept | Statement |
|---|---|
| FWER | $P(\text{any false positive})$ — Bonferroni divides $\alpha$ by $m$; conservative, protects primary/OEC-style decisions |
| FDR | $E[\text{false discoveries}/\text{discoveries}]$ — Benjamini-Hochberg; more power, suited to exploratory scans |
| Peeking is a different problem | Multiplicity across *metrics/segments* (this notebook) vs. across *time* (notebook 08) require different corrections |

## References

- Benjamini, Y., & Hochberg, Y. (1995). Controlling the False Discovery Rate: A Practical and Powerful Approach to Multiple Testing. *Journal of the Royal Statistical Society, Series B*, 57(1), 289-300.
- Kohavi, R., Tang, D., & Xu, Y. (2020). *Trustworthy Online Controlled Experiments*, Ch. 17 (multiple comparisons in practice).
- See `08_peeking_sequential_risk.ipynb` for always-valid inference (the sequential/temporal version of multiplicity).
